# Adaptation study: subject-adaptive pseudo-labeling (run-3 file simulation)

In [ ]:
import os, warnings, time
import numpy as np
import pandas as pd
from sklearn.covariance import LedoitWolf
warnings.filterwarnings("ignore")
np.set_printoptions(suppress=True)

data_root = "/kaggle/input/competitions/low-cost-motor-imagery-decoding-for-rehab-cross-subject"
train_dir = os.path.join(data_root, "Training set")
t0 = time.time()

In [ ]:
import numpy as np
import pandas as pd
import glob, os, re
from scipy import signal

FS = 250.0
CHANNELS = ['Fz', 'C3', 'Cz', 'C4', 'PO7', 'Pz', 'PO8', 'Oz']
CUE_LEN = 5.0
BASE_LEN = 3.0
_NOTCH_CACHE = [None]
def notch(x):
    if _NOTCH_CACHE[0] is None:
        _NOTCH_CACHE[0] = signal.iirnotch(50.0 / (FS / 2.0), 30.0)
    b, a = _NOTCH_CACHE[0]
    return signal.filtfilt(b, a, x, axis=0)

def car(x):
    return x - x.mean(axis=1, keepdims=True)

def read_epochs(path, labeled=True):
    df = pd.read_csv(path, low_memory=False)
    x = df[CHANNELS].to_numpy(np.float64)
    m = df['Marker_val'].astype(str)
    x = car(x)
    x = notch(x)
    if labeled:
        cue_idx = np.flatnonzero(m.isin(['rest', 'move']).to_numpy())
        labels = np.array([1 if v == 'move' else 0 for v in m.iloc[cue_idx]])
    else:
        cue_idx = np.flatnonzero((m == 'cue_start').to_numpy())
        labels = None
    tcue = int(CUE_LEN * FS)
    tbase = int(BASE_LEN * FS)
    cues, bases = [], []
    for i in cue_idx:
        s, e = i, i + tcue
        if e > len(x):
            continue
        cues.append(x[s:e])
        b0 = max(0, i - tbase)
        bases.append(x[b0:i][-tbase:])
    return np.stack(cues), np.stack(bases), labels


In [ ]:
BANDS = [(4, 8), (8, 12), (12, 16), (16, 20), (20, 25), (25, 30)]
_BUTTER = {}
def _butter(l, h, order=4):
    key = (float(l), float(h))
    if key not in _BUTTER:
        nyq = FS / 2
        _BUTTER[key] = signal.butter(order, [max(0.01, l / nyq), min(0.99, h / nyq)], btype='bandpass')
    return _BUTTER[key]

def band_filter(x, l, h):
    b, a = _butter(l, h)
    axis = 1 if x.ndim == 3 else 0
    return signal.filtfilt(b, a, x, axis=axis)

def band_power_features(cue, base, bands=BANDS, off=0.0):
    if off > 0:
        k = int(off * FS)
        cue = cue[:, k:, :]
    N = cue.shape[0]
    feats = []
    for (l, h) in bands:
        cf = band_filter(cue, l, h)
        bf = band_filter(base, l, h)
        pc = np.log(np.maximum((cf ** 2).mean(axis=1), 1e-12))
        pb = np.log(np.maximum((bf ** 2).mean(axis=1), 1e-12))
        feats.append(pc)
        feats.append(np.log(np.maximum((cf ** 2).mean(axis=1), 1e-12) / np.maximum((bf ** 2).mean(axis=1), 1e-12)))
    return np.concatenate(feats, axis=1)

def _spdm(cue):
    return np.array([np.cov(x.T) for x in cue])

def _matrix_log(A):
    w, v = np.linalg.eigh(A)
    return v @ np.diag(np.log(np.maximum(w, 1e-12))) @ v.T

def _matrix_op(A, p):
    w, v = np.linalg.eigh(A)
    return v @ np.diag(w ** p) @ v.T

def riemannian_features(cue, bands=BANDS, ref=None, shrink=True, off=0.0):
    if off > 0:
        k = int(off * FS)
        cue = cue[:, k:, :]
    est = LedoitWolf()
    covs = []
    for (l, h) in bands:
        cf = band_filter(cue, l, h)
        covs.append(np.array([est.fit(x).covariance_ for x in cf]))
    if ref is None:
        ref = np.stack([np.mean(c, axis=0) for c in covs])
    feats = []
    for ci, r in zip(covs, ref):
        Ginv_sqrt = _matrix_op(r, -0.5)
        f = []
        for C in ci:
            W = Ginv_sqrt @ C @ Ginv_sqrt
            L = _matrix_log(W)
            idx = np.triu_indices(8)
            v = L[idx]
            v[1:] *= np.sqrt(2.0)
            f.append(v)
        feats.append(np.stack(f))
    return np.concatenate(feats, axis=1), ref

def csp_features(cue, y, bands=((8, 12), (12, 16)), k=4, off=0.0):
    if off > 0:
        k2 = int(off * FS)
        cue = cue[:, k2:, :]
    est = LedoitWolf()
    filters = []
    feats = []
    for (l, h) in bands:
        cf = band_filter(cue, l, h)
        C1 = np.mean([np.cov(x.T) for x in cf[y == 1]], axis=0) if np.sum(y == 1) > 2 else np.eye(8)
        C0 = np.mean([np.cov(x.T) for x in cf[y == 0]], axis=0) if np.sum(y == 0) > 2 else np.eye(8)
        C1, C0 = est.fit(C1).covariance_, est.fit(C0).covariance_
        w, V = np.linalg.eigh(np.linalg.pinv(C0) @ C1)
        order = np.argsort(w)
        V = V[:, order]
        F = np.concatenate([V[:, -k:], V[:, :k]], axis=1)
        filters.append(F)
        proj = [x @ F for x in cf]
        d = np.log(np.maximum(np.array([np.diag(np.cov(p.T)) for p in proj]), 1e-12))
        feats.append(d)
    return np.concatenate(feats, axis=1), filters

def csp_transform(cue, filters, bands=((8, 12), (12, 16)), off=0.0):
    if off > 0:
        k2 = int(off * FS)
        cue = cue[:, k2:, :]
    feats = []
    for (l, h), F in zip(bands, filters):
        cf = band_filter(cue, l, h)
        proj = [x @ F for x in cf]
        d = np.log(np.maximum(np.array([np.diag(np.cov(p.T)) for p in proj]), 1e-12))
        feats.append(d)
    return np.concatenate(feats, axis=1)


In [ ]:
# load all train files, keep run id
recs = []
for f in sorted(glob.glob(os.path.join(train_dir, "*.csv"))):
    c, b, lab = read_epochs(f, labeled=True)
    subj = re.search(r"(S\d+)-", os.path.basename(f)).group(1)
    run = os.path.basename(f).split("-")[1][:3]
    for i in range(len(lab)):
        recs.append((subj, run, c[i], b[i], lab[i]))
cues = np.stack([r[2] for r in recs]); bases = np.stack([r[3] for r in recs])
y = np.array([r[4] for r in recs]); sub = np.array([r[0] for r in recs]); run = np.array([r[1] for r in recs])
subjects = sorted(set(sub))
print("loaded", cues.shape, "subjects", len(subjects), round(time.time()-t0, 1), "s")
bp_all = band_power_features(cues, bases)
print("bp", bp_all.shape, round(time.time()-t0, 1), "s")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans

def w3_fold(tr, va, y_tr):
    rts_tr, ref = riemannian_features(cues[tr])
    cspf_tr, f0 = csp_features(cues[tr], y_tr)
    rts_va, _ = riemannian_features(cues[va], ref=ref)
    cspf_va = csp_transform(cues[va], f0)
    return (np.concatenate([bp_all[tr], rts_tr, cspf_tr], 1),
            np.concatenate([bp_all[va], rts_va, cspf_va], 1))

def crosssub_proba(va, tr, C=0.1):
    Ftr, Fva = w3_fold(tr, va, y[tr])
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
    m.fit(Ftr, y[tr])
    return m.predict_proba(Fva)[:, 1]

def adapt_predict(xv, P0, iters=1, blend=0.0, kind="lda"):
    """xv: per-subject z-scored features (n, F); P0: cross-subject proba."""
    Pv = P0.copy()
    for it in range(iters):
        km = KMeans(2, n_init=10, random_state=0).fit(xv)
        lab = km.labels_
        d = P0[lab == 0].mean() > P0[lab == 1].mean()
        plab = (lab == (0 if d else 1)).astype(int)
        if kind == "lda":
            try:
                m = LinearDiscriminantAnalysis(solver="lsqr").fit(xv, plab)
                Pv = m.predict_proba(xv)[:, 1]
            except Exception:
                Pv = plab.astype(float)
        elif kind == "csp":
            # per-subject CSP on raw band-passed windows is handled outside
            Pv = plab.astype(float)
        P0 = plab.astype(float) if it < iters - 1 else P0
    return (1 - blend) * Pv + blend * P0

def subj_z(bp, va):
    mu = bp[va].mean(axis=0); sd = bp[va].std(axis=0) + 1e-8
    return (bp[va] - mu) / sd

In [ ]:
# ---------- RUN-3 FILE SIMULATION ----------
# For each subject: treat its run-3 trials as an unlabeled "test file".
# Cross-subject model uses the other 16 subjects (all runs).
print("=== run-3 simulation: cross-subject baseline ===")
accs_base = []
for s in subjects:
    va = (sub == s) & (run == "003")
    tr = ~va
    if va.sum() == 0:
        continue
    P0 = crosssub_proba(va, tr)
    accs_base.append(((P0 > 0.5).astype(int) == y[va]).mean())
print("baseline run3-file LOSO:", np.round(accs_base, 3), "mean %.4f" % np.mean(accs_base))

In [ ]:
print("=== adaptation variants (run-3 file simulation) ===")
variants = [
    ("adapt1_lda_blend0", 1, 0.0, "lda"),
    ("adapt1_lda_blend0.5", 1, 0.5, "lda"),
    ("adapt1_lda_blend0.8", 1, 0.8, "lda"),
    ("adapt3_lda_blend0", 3, 0.0, "lda"),
    ("adapt3_lda_blend0.5", 3, 0.5, "lda"),
]
res = {}
for name, iters, blend, kind in variants:
    accs = []
    for s in subjects:
        va = (sub == s) & (run == "003")
        tr = ~va
        if va.sum() == 0:
            continue
        P0 = crosssub_proba(va, tr)
        xv = subj_z(bp_all, va)
        Pf = adapt_predict(xv, P0, iters=iters, blend=blend, kind=kind)
        accs.append(((Pf > 0.5).astype(int) == y[va]).mean())
    res[name] = np.mean(accs)
    print(name, np.round(accs, 3), "mean %.4f" % np.mean(accs))

In [ ]:
# also simulate with ALL runs of the subject as the unlabeled pool (more data)
print("=== adaptation, pool = all 3 runs of subject ===")
for name, iters, blend, kind in variants[:3]:
    accs = []
    for s in subjects:
        va = sub == s
        tr = ~va
        P0 = crosssub_proba(va, tr)
        xv = subj_z(bp_all, va)
        Pf = adapt_predict(xv, P0, iters=iters, blend=blend, kind=kind)
        accs.append(((Pf > 0.5).astype(int) == y[va]).mean())
    print(name + " pool=all3", "mean %.4f" % np.mean(accs))
print("STUDY DONE", round(time.time() - t0, 1), "s")